# Week 10 · Topic 2 — Inserting and Querying

**Box we are building: STORE, safely**

Adding rows, getting them back, and the one habit that keeps you safe from
SQL injection.

> **Already know this?** Skip to the *Topic 2 Challenge*.

> **Setup.** This notebook can be run on its own. The next cell connects to the same `network.db` file as Topic 1 and makes sure the `devices` table exists. It is safe to run more than once.

In [6]:
import sqlite3

conn = sqlite3.connect("network.db")
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS devices (
        id     INTEGER PRIMARY KEY,
        name   TEXT NOT NULL,
        ip     TEXT UNIQUE NOT NULL,
        active INTEGER DEFAULT 1,
        type   TEXT
    )
""")
conn.commit()

---
## 1. `?` placeholders — never build SQL by joining strings

In [7]:
cursor.execute(
    "INSERT INTO devices (name, ip, type) VALUES (?, ?, ?)",
    ("Router-A", "10.0.0.1", "router"),
)
conn.commit()

### Your turn
Insert a device of your own using a placeholder query.

In [8]:
cursor.execute(
    "INSERT INTO devices (name, ip, type) VALUES (?, ?, ?)",
    ("My-Laptop", "192.168.1.50", "computer")
)
conn.commit()

---
## 2. `executemany()` — several rows at once

In [ ]:
rows = [("Switch-1", "10.0.0.2", "switch"), ("AP-Lobby", "10.0.0.3", "access point")]
cursor.executemany("INSERT INTO devices (name, ip, type) VALUES (?, ?, ?)", rows)
conn.commit()

### Your turn
Use `executemany()` to insert 3 rows of your own choosing.

In [9]:
my_devices = [
    ("Server-01", "10.0.0.10", "server"),
    ("Printer-A", "10.0.0.11", "printer"),
    ("Phone-John", "10.0.0.12", "mobile")
]

cursor.executemany(
    "INSERT INTO devices (name, ip, type) VALUES (?, ?, ?)", 
    my_devices
)

conn.commit()

---
## 3. `fetchall()`, `fetchone()`, and iterating the cursor directly

In [10]:
print(cursor.execute("SELECT * FROM devices").fetchall())
print(cursor.execute("SELECT name FROM devices WHERE id = ?", (999,)).fetchone())   # None, no crash

for name, ip in cursor.execute("SELECT name, ip FROM devices"):
    print(name, "->", ip)

[(1, 'Router-A', '10.0.0.1', 1, 'router'), (2, 'My-Laptop', '192.168.1.50', 1, 'computer'), (3, 'Server-01', '10.0.0.10', 1, 'server'), (4, 'Printer-A', '10.0.0.11', 1, 'printer'), (5, 'Phone-John', '10.0.0.12', 1, 'mobile')]
None
Router-A -> 10.0.0.1
My-Laptop -> 192.168.1.50
Server-01 -> 10.0.0.10
Printer-A -> 10.0.0.11
Phone-John -> 10.0.0.12


### Your turn
SELECT a row you know does not exist and confirm `fetchone()` gives `None`, not an error.

In [11]:
row = cursor.execute("SELECT * FROM devices WHERE id = ?", (9999,)).fetchone()

print(row)

None


---
## 4. SQL injection — seeing it actually work, and actually get blocked

Building SQL by joining a string in is not a style choice — it is a real
vulnerability.

In [ ]:
name = "' OR '1'='1"
unsafe = f"SELECT * FROM devices WHERE name = '{name}'"
print(cursor.execute(unsafe).fetchall())    # returns EVERY row - the WHERE did nothing

safe = cursor.execute("SELECT * FROM devices WHERE name = ?", (name,)).fetchall()
print(safe)    # empty - treated as data, not code

### Your turn
Try your own malicious-looking string against both the unsafe and safe versions and compare the results.

In [ ]:
# your code here

---
## 5. Counting and summarising: `COUNT`, `AVG`, `GROUP BY`

`SELECT` can answer questions about a whole table, not only list its rows. The dashboard you build in Week 11 is made of questions like these.

| Question | SQL |
|---|---|
| how many rows? | `SELECT COUNT(*) FROM games` |
| average, highest, lowest | `SELECT AVG(rating), MAX(rating), MIN(rating) FROM games` |
| how many in each group? | `SELECT genre, COUNT(*) FROM games GROUP BY genre` |
| name a result column | `COUNT(*) AS how_many` |
| groups, largest first | `... GROUP BY genre ORDER BY how_many DESC` |
| part of a word | `WHERE title LIKE ?` with the value `"%craft%"` (`%` means any characters) |

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE games (title TEXT, genre TEXT, year INTEGER, rating REAL)")
conn.executemany("INSERT INTO games VALUES (?, ?, ?, ?)", [
    ("Hollow Knight", "Platformer", 2017, 9.1),
    ("Celeste", "Platformer", 2018, 8.9),
    ("Portal 2", "Puzzle", 2011, 9.5),
    ("Tetris", "Puzzle", 1984, 9.0),
    ("Hades", "Roguelike", 2020, 9.3),
])

print(conn.execute("SELECT COUNT(*), AVG(rating), MAX(rating) FROM games").fetchone())

query = """SELECT genre, COUNT(*) AS how_many, AVG(rating) AS average_rating
           FROM games GROUP BY genre ORDER BY how_many DESC"""
for row in conn.execute(query):
    print(row)

print(conn.execute("SELECT title FROM games WHERE title LIKE ?", ("%e%",)).fetchall())

### Your turn
Write a query that gives the highest rating in each genre.

In [ ]:
# your code here

---
## 6. Rules in the table: `UNIQUE`, `NOT NULL` and `COLLATE NOCASE`

A column can carry rules, and the database enforces them for you. Your `users` table uses three:

```sql
username TEXT NOT NULL UNIQUE COLLATE NOCASE
```

- `NOT NULL`: a row must have a value here.
- `UNIQUE`: no two rows can have the same value. Breaking this raises `sqlite3.IntegrityError`.
- `COLLATE NOCASE`: comparisons ignore capital letters, so `Anna` and `anna` count as the same name.

This means `register()` does not have to search for a duplicate first. It tries the `INSERT` and catches the error.

In [ ]:
conn.execute("CREATE TABLE users (username TEXT NOT NULL UNIQUE COLLATE NOCASE)")
conn.execute("INSERT INTO users VALUES (?)", ("Anna",))

try:
    conn.execute("INSERT INTO users VALUES (?)", ("anna",))
except sqlite3.IntegrityError as error:
    print("Refused:", error)

print(conn.execute("SELECT * FROM users WHERE username = ?", ("ANNA",)).fetchall())

### Your turn
Try inserting `None` as a username and read the error.

In [ ]:
# your code here

`examples/03_counting_and_grouping.py` contains the grouping queries in a single script.

---
## Topic 2 Challenge

Insert 4 devices of different types. Then write one query using `WHERE`,
`ORDER BY` and `LIMIT` together to find the 2 most recently inserted routers.

In [ ]:
# your code here

---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | inserting cleaned rows from a DataFrame, one at a time or via `to_sql()` |
| **Cyber Security** | querying incidents `WHERE severity = ?` safely, even with attacker-controlled input |
| **IT** | filtering devices `WHERE type = ? AND active = 1` |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| building SQL with an f-string and a real value | SQL injection — a crafted value can return or destroy data it should not | always use `?` placeholders |
| forgetting the trailing comma on a single value | `ProgrammingError: Incorrect number of bindings supplied` | `(value,)` is a tuple; `(value)` is not |
| expecting `fetchone()` to error when nothing matches | it returns `None` quietly | check for `None` before using the result |
| mixing up `fetchall()` and iterating the cursor | both work, but calling `fetchall()` twice returns nothing the second time | fetch once, store the result if you need it twice |
| selecting a plain column together with `COUNT(*)` but no `GROUP BY` | one meaningless row | add `GROUP BY` for the plain column |
| `LIKE '%?%'` | the placeholder is not replaced | pass the whole pattern as the value: `("%" + text + "%",)` |

## Go deeper

`examples/01_insert_and_select.py` — INSERT, executemany, fetchall, fetchone
`examples/02_injection_filtering_sorting.py` — a real injection, WHERE, AND, ORDER BY, LIMIT
Docs: [sqlite3 — Python docs](https://docs.python.org/3/library/sqlite3.html)

---
## Recap

| You can now | Using |
|---|---|
| insert a row safely | `INSERT ... VALUES (?, ?, ?)` |
| insert many at once | `executemany()` |
| get rows back | `fetchall()`, `fetchone()` |
| filter, sort, limit | `WHERE`, `ORDER BY`, `LIMIT` |
| stay safe from injection | `?` placeholders, always |
| summarise a table | `COUNT`, `AVG`, `MAX`, `GROUP BY`, `AS` |
| search for part of a word | `LIKE ?` with `"%word%"` |
| let the table enforce rules | `UNIQUE`, `NOT NULL`, `COLLATE NOCASE` |

**Next:** Topic 3 — changing and removing rows, safely.